In [ ]:
import os
import re
import json
import time
import pickle
from pathlib import Path

import numpy as np
from thefuzz import fuzz
from langchain_openai import OpenAIEmbeddings

PREDICTIONS_DIR = Path("./result-150")
GROUND_TRUTH_PATH = Path("./ground_truth.json")
OUTPUT_JSON = Path("./table4_results.json")
CACHE_PATH = Path("./embedding_cache.pkl")

EMBED_MODEL = "qwen/qwen3-embedding-8b"
BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "sk-or-v1-xxx")

CATEGORIES = ["catalyst", "co-catalyst", "light source", "lamp",
              "reactor type", "reaction medium", "operation mode"]

DISPLAY = {
    "catalyst": "Catalyst",
    "co-catalyst": "Co-Catalyst",
    "light source": "Light Source",
    "lamp": "Lamp",
    "reactor type": "Reactor Type",
    "reaction medium": "Reaction Medium",
    "operation mode": "Operation Mode",
}

In [2]:
with open(GROUND_TRUTH_PATH) as f:
    ground_truth = json.load(f)

print(f"Loaded ground truth for {len(ground_truth)} papers")

Loaded ground truth for 1096 papers


In [3]:
if not API_KEY:
    raise RuntimeError("Set OPENROUTER_API_KEY in your environment before running.")

if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        embedding_cache = pickle.load(f)
else:
    embedding_cache = {}

embedder = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=BASE_URL,
    api_key=API_KEY,
    check_embedding_ctx_length=False,
)

def embed_texts(texts):
    missing = sorted({t for t in texts if t and t not in embedding_cache})
    for start in range(0, len(missing), 64):
        batch = missing[start:start + 64]
        vectors = None
        for attempt in range(5):
            try:
                vectors = embedder.embed_documents(batch)
                break
            except Exception:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        for text, vector in zip(batch, vectors):
            embedding_cache[text] = np.asarray(vector, dtype=np.float32)
    if missing:
        with open(CACHE_PATH, "wb") as f:
            pickle.dump(embedding_cache, f)
    return {t: embedding_cache[t] for t in texts if t in embedding_cache}

In [4]:
def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def lexical_ratio(reference, prediction):
    return fuzz.partial_ratio(str(reference), str(prediction)) / 100.0

def canon_config(name):
    tokens = [t for t in re.split(r"[^a-z0-9]+", name.lower()) if t]
    chunk_map = {"naive": "Naive", "recursive": "Recursive", "semantic": "Semantic"}
    retr_map = {"naive": "Naive", "hybrid": "Hybrid", "rerank": "Rerank",
                "reranking": "Rerank", "reranker": "Rerank", "ensemble": "Hybrid"}
    chunking = None
    retrieval = None
    for token in tokens:
        if chunking is None and token in chunk_map:
            chunking = chunk_map[token]
            continue
        if retrieval is None and token in retr_map:
            retrieval = retr_map[token]
    if chunking and retrieval:
        return f"{chunking}-{retrieval}"
    return name

def evaluate_config(config_dir):
    cos_sums = {c: 0.0 for c in CATEGORIES}
    ratio_sums = {c: 0.0 for c in CATEGORIES}
    counts = {c: 0 for c in CATEGORIES}

    texts = set()
    records = []
    for path in sorted(config_dir.glob("annotated_annotation_*.json")):
        match = re.search(r"(\d+)$", path.stem)
        if match is None:
            continue
        paper_id = match.group(1)
        if paper_id not in ground_truth:
            continue
        with open(path) as f:
            prediction = json.load(f)
        gold = ground_truth[paper_id]
        for category in CATEGORIES:
            reference = gold.get(category)
            generated = prediction.get(category)
            if not reference or not generated:
                continue
            reference = str(reference).strip()
            generated = str(generated).strip()
            if not reference or not generated:
                continue
            records.append((category, reference, generated))
            texts.add(reference)
            texts.add(generated)

    vectors = embed_texts(list(texts))

    for category, reference, generated in records:
        if reference not in vectors or generated not in vectors:
            continue
        cos_sums[category] += cosine_similarity(vectors[reference], vectors[generated])
        ratio_sums[category] += lexical_ratio(reference, generated)
        counts[category] += 1

    cos_cell = {}
    ratio_cell = {}
    for category in CATEGORIES:
        if counts[category] == 0:
            cos_cell[category] = 0.0
            ratio_cell[category] = 0.0
        else:
            cos_cell[category] = cos_sums[category] / counts[category]
            ratio_cell[category] = ratio_sums[category] / counts[category]

    cos_avg = sum(cos_cell.values()) / len(CATEGORIES)
    ratio_avg = sum(ratio_cell.values()) / len(CATEGORIES)
    return cos_cell, cos_avg, ratio_cell, ratio_avg

In [5]:
config_dirs = sorted(p for p in PREDICTIONS_DIR.iterdir() if p.is_dir())

results = {}
for config_dir in config_dirs:
    label = canon_config(config_dir.name)
    cos_cell, cos_avg, ratio_cell, ratio_avg = evaluate_config(config_dir)
    results[label] = {
        "cos_sim": {DISPLAY[c]: round(cos_cell[c], 4) for c in CATEGORIES},
        "ratio": {DISPLAY[c]: round(ratio_cell[c], 4) for c in CATEGORIES},
    }
    results[label]["cos_sim"]["Average"] = round(cos_avg, 4)
    results[label]["ratio"]["Average"] = round(ratio_avg, 4)
    print(f"{label}: cos_sim avg {cos_avg:.4f}, ratio avg {ratio_avg:.4f}")

.ipynb_checkpoints: cos_sim avg 0.0000, ratio avg 0.0000
Naive-Hybrid: cos_sim avg 0.7902, ratio avg 0.6170
Naive-Naive: cos_sim avg 0.7979, ratio avg 0.6282
Naive-Rerank: cos_sim avg 0.7451, ratio avg 0.5461
Recursive-Hybrid: cos_sim avg 0.7870, ratio avg 0.6142
Recursive-Naive: cos_sim avg 0.7981, ratio avg 0.6321
Recursive-Rerank: cos_sim avg 0.7447, ratio avg 0.5445
Semantic-Hybrid: cos_sim avg 0.7897, ratio avg 0.6179
Semantic-Naive: cos_sim avg 0.7934, ratio avg 0.6285
Semantic-Rerank: cos_sim avg 0.7509, ratio avg 0.5556


In [ ]:
print(json.dumps(results, indent=2))

In [ ]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {OUTPUT_JSON}")